In [3]:
import os
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
import pickle
from sklearn.preprocessing import LabelEncoder
from torchvision import models, transforms

In [29]:
df = pd.read_csv("../data/processed/spectrograms/manifest.csv")

output_dir = "../data/processed/spectrograms/"
os.makedirs(output_dir, exist_ok=True)

# manter apenas arquivos válidos
df = df[df["image_path"].apply(os.path.exists)].reset_index(drop=True)

print("Total de amostras:", len(df))
df.head()

Total de amostras: 30166


,image_path,label,audioSource,roi_start,roi_end,roi_min_freq,roi_max_freq,roi_duration
0,../data/processed/spectrograms\images\Megascop...,Megascops choliba,W04856768S2013814_20230811_033000.WAV,38.071627,41.774503,0.601953,1.266653,3.702877
1,../data/processed/spectrograms\images\Leptotil...,Leptotila verreauxi,W04856768S2013814_20230811_072000.WAV,6.038564,7.289066,0.373524,0.554806,1.250502
2,../data/processed/spectrograms\images\Leptotil...,Leptotila verreauxi,W04856768S2013814_20230811_072000.WAV,9.049033,10.067961,0.334678,0.541857,1.018928
3,../data/processed/spectrograms\images\Leptotil...,Leptotila verreauxi,W04856768S2013814_20230811_072000.WAV,19.099367,20.396185,0.347627,0.541857,1.296817
4,../data/processed/spectrograms\images\Leptotil...,Leptotila verreauxi,W04856768S2013814_20230811_072000.WAV,24.332952,25.815029,0.321729,0.554806,1.482077


In [30]:
# =========================
# DEVICE E TRANSFORM
# =========================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# CNN → Global Average Pooling → vetor fixo
transform = transforms.Compose([
    transforms.ToTensor(),          # Cada imagem vira um tensor - (3, H, W)
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], # Média e desvio padrão da ImageNet
        std=[0.229, 0.224, 0.225],
    ),
])


In [31]:
# =========================
# INICIALIZAÇÃO DOS MODELOS 
# =========================

def build_model(model_name):

    if model_name == "resnet":
        weights = models.ResNet50_Weights.DEFAULT
        model = models.resnet50(weights=weights)

        # remove classificador final
        model.fc = torch.nn.Identity()

        feature_dim = 2048

    elif model_name == "efficientnet":
        weights = models.EfficientNet_B2_Weights.DEFAULT
        model = models.efficientnet_b2(weights=weights)

        # remove classificador final
        model.classifier = torch.nn.Identity()

        feature_dim = 1408

    else:
        raise ValueError("Modelo inválido")

    # Move o modelo para GPU (CUDA) ou CPU
    model = model.to(DEVICE)
    # Coloca a rede em modo de inferência
    model.eval()               

    return model, feature_dim

In [32]:
# =========================
# EXTRAÇÃO DE FEATURES
# =========================

def extract_features(df, model):

    features = []

    with torch.no_grad():

        for _, row in tqdm(df.iterrows(), total=len(df)):

            img = Image.open(row["image_path"]).convert("RGB")
            x = transform(img)
            x = x.unsqueeze(0).to(DEVICE)

            f = model(x)
            f = f.cpu().numpy().flatten() #flatten: (1, C) → (C,)

            extra = np.array([
                row["roi_min_freq"],
                row["roi_max_freq"],
                row["roi_duration"],
            ], dtype=np.float32)

            f_final = np.concatenate([f, extra])

            features.append(f_final)

    return np.array(features)

### Extração Features ResNet

In [34]:
model_resnet, dim_resnet = build_model("resnet")

X_resnet = extract_features(df, model_resnet)
np.savez_compressed(os.path.join(output_dir, "features_resnet.npz"), X_resnet)

print("Shape ResNet:", X_resnet.shape)
print("NaNs:", np.isnan(X_resnet).sum())
print("Infs:", np.isinf(X_resnet).sum())

100%|██████████| 30166/30166 [58:22<00:00,  8.61it/s]     


Shape ResNet: (30166, 2051)
NaNs: 0
Infs: 0


In [33]:
print(X_resnet.shape)
print(X_resnet.nbytes / 1e6, "MB")

(30166, 2051)
247.481864 MB


### Extração Features EfficientNet B2

In [35]:
model_eff, dim_eff = build_model("efficientnet")

X_eff = extract_features(df, model_eff)
np.savez_compressed(os.path.join(output_dir, "features_efficientnet.npz"), X_eff)

print("Shape EfficientNet:", X_eff.shape)
print("NaNs:", np.isnan(X_eff).sum())
print("Infs:", np.isinf(X_eff).sum())

100%|██████████| 30166/30166 [11:27<00:00, 43.89it/s]


Shape EfficientNet: (30166, 1411)
NaNs: 0
Infs: 0


In [9]:
X_efficientnet_loaded = np.load("../data/processed/spectrograms/features_efficientnet.npz")

In [10]:
X = X_efficientnet_loaded["arr_0"]
print(X.shape)
print(X.nbytes / 1e6, "MB")

(30166, 1411)
170.256904 MB


### Salvando Labels e LabelEncoder

In [36]:
encoder = LabelEncoder()
y = encoder.fit_transform(df["label"])

np.save(os.path.join(output_dir, "labels.npy"), y)

with open(os.path.join(output_dir, "label_encoder.pkl"), "wb") as f:
    pickle.dump(encoder, f)